# Biomarker S6 — Biomarker neo bề mặt xương + FCL (mất sụn toàn bề dày)

**Input:** `MASK_DIR` (S2) + `cohort_manifest.csv` (S1/S2, có `KL`, `subject`, `source_dataset`).
**Output (thư mục MỚI, không đụng bảng cũ):** `knee_biomarkers/s6_fcl/biomarker_table_v2.csv`.

Bảng v2 = **superset** của `biomarker_table.csv` (S3): giữ nguyên các cột cũ (`vol_*`, `thickness_*`,
`denuded_ratio_*`, `extrusion_*`, cùng định nghĩa, cùng số — mục 4 đối chiếu 1:1) và thêm họ biomarker
đo trên **lưới bề mặt xương** (3D, không theo lát):

| Cột | Nghĩa | Thuật ngữ Eckstein/Wirth |
|---|---|---|
| `tab_<c>_mm2` | diện tích footprint mảng sụn trên bề mặt xương | tAB |
| `cab_<c>_mm2` | phần footprint còn sụn | cAB |
| `fcl_<c>_mm2`, `fcl_<c>_pct` | **mất sụn toàn bề dày**: footprint mà độ dày = 0 | dAB, dAB% |
| `thc_tab_<c>_mm` | độ dày trung bình trên tAB, vùng mất sụn tính = 0 | ThC.tAB |
| `thc_cab_<c>_mm` | độ dày trung bình chỉ trên vùng còn sụn | ThC.cAB |
| `thickp05_<c>_mm` | phân vị 5% độ dày trên cAB | |
| `thin_le05_<c>_pct`, `thin_le10_<c>_pct` | % tAB có sụn mỏng ≤ 0.5 / ≤ 1.0 mm | partial-thickness |
| `fcl_<c>_ndef`, `fcl_<c>_maxdef_mm2` | số ổ mất sụn (≥ 5 mm²) và ổ lớn nhất | |

`<c>` ∈ {`fem`, `fem_med`, `fem_lat`, `mt`, `lt`}. Không có sụn bánh chè vì mask không có xương bánh chè.

**Cách đo** (một định nghĩa duy nhất trong `bsc/biomarkers.py`, có 14 test phantom ở đúng spacing thật):
1. Bề mặt xương = marching cubes trên mask xương làm mượt 0.5 mm; pháp tuyến hướng ra ngoài, tự kiểm bằng dữ liệu.
2. Tại mỗi đỉnh bắn tia theo pháp tuyến: độ dày sụn = đoạn sụn liên tục đầu tiên (bước 0.1 mm, tối đa 6 mm,
   cho phép khe ≤ 1 mm giữa xương và sụn).
3. Footprint = **closing trắc địa** bán kính `close_mm` của vùng có sụn → lấp lỗ *bên trong* mảng sụn.
   Xương đùi closing riêng từng nửa trong/ngoài (mặt phẳng suy từ trọng tâm hai sụn chày) để không bắc cầu qua hõm liên lồi cầu.
4. FCL = đỉnh trong footprint mà độ dày = 0; diện tích = tổng diện tích barycentric.

**Khác gì `denuded_ratio` cũ:** mẫu số là footprint mảng sụn, không phải toàn bộ bề mặt xương trong FOV; tách được khoang.
`thc_tab` giảm *đúng bằng* phần diện tích mất, trong khi `thickness_*` cũ (thể tích / tiếp xúc) gần như mù với mất sụn —
đã chứng minh bằng phantom (`tests/test_biomarkers.py`).

**Hạn chế phải nhớ khi đọc số:**
- Closing chỉ lấp lỗ *được bao quanh* bởi sụn còn lại. Mất sụn ở rìa mảng hoặc cả khoang trơ trụi **không** được đếm → FCL là **cận dưới**.
  Bán kính closing là tham số nhạy (mục 6 đo độ nhạy), phải báo cáo giá trị dùng.
- Không có nhãn gai xương: mũ sụn gai xương có thể kéo footprint ra rìa (mục 7).
- Chưa có cặp GT/AI cho cùng ca ở đây nên chưa đo được độ tin cậy của FCL từ mask AI. Việc đó cần notebook riêng trên 103 ca test OAI-ZIB.

In [ ]:
# ============================================================
# Moi truong: Drive + clone repo. MOI biomarker/model co MOT dinh nghia trong bsc/*.py,
# notebook chi goi - khong copy code vao day (quy tac "mot dinh nghia" cua CLAUDE.md).
# ============================================================
!pip install -q SimpleITK scikit-image scipy pandas matplotlib tqdm 2>/dev/null
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

from bsc import biomarkers as BM
print("bsc.biomarkers:", BM.__file__)
print("legacy prefixes :", BM.LEGACY_PREFIXES)
print("surface prefixes:", BM.SURFACE_PREFIXES)

## 0) Cấu hình — thư mục MỚI `s6_fcl/`, mask cũ chỉ đọc

In [ ]:
from pathlib import Path
import time, json
import numpy as np, pandas as pd, SimpleITK as sitk
from tqdm.auto import tqdm

BIOM_DIR = Path("/content/drive/MyDrive/knee_biomarkers")   # cua S1..S5 - CHI DOC
MASK_DIR = BIOM_DIR / "masks"
COHORT_CSV = BIOM_DIR / "cohort_manifest.csv"
S3_CSV = BIOM_DIR / "biomarker_table.csv"                     # de doi chieu cot legacy (muc 4)

OUT_DIR = BIOM_DIR / "s6_fcl"                                  # MOI - khong ghi de gi cua S3
OUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR = OUT_DIR / "qc"; QC_DIR.mkdir(exist_ok=True)
OUT_CSV = OUT_DIR / "biomarker_table_v2.csv"
CHECKPOINT_CSV = OUT_DIR / "biomarker_table_v2_partial.csv"
assert str(OUT_DIR).startswith("/content/drive/"), "Drive-first: khong ghi vao /content/"

# Tham so do - luu params.json canh bang de biet bang nay do voi gia tri nao
PARAMS = dict(close_mm=8.0, step_mm=0.1, max_mm=6.0, gap_mm=1.0, smooth_mm=0.5,
              min_island_mm2=100.0, min_defect_mm2=5.0)
SAVE_EVERY = 10
MAX_CASES_THIS_RUN = None       # vd 50 de chay thu mot phan; None = tat ca ca con thieu
json.dump(PARAMS, open(OUT_DIR / "params.json", "w"), indent=2)

cohort = pd.read_csv(COHORT_CSV)
n0 = len(cohort)
cohort = cohort.drop_duplicates(subset=["case_id"]).reset_index(drop=True)
if len(cohort) < n0:
    print(f"CANH BAO: cohort_manifest.csv co {n0 - len(cohort)} dong trung case_id, da loai")
mask_files = {p.name.replace(".nii.gz", ""): p for p in MASK_DIR.glob("*.nii.gz")}
cohort = cohort[cohort["case_id"].isin(mask_files)].reset_index(drop=True)
print("ca co mask:", len(cohort))
print(cohort["KL"].value_counts().sort_index())
if "source_dataset" in cohort.columns:
    print(cohort["source_dataset"].value_counts())

## 1) Đọc mask — spacing lấy từ header, theo đúng thứ tự trục của mảng

`io_utils.py` đã ghi nhận mask thật khi đọc ra có trục 0.70 mm nằm **cuối**, không phải đầu như `core.SPACING`.
Module không có spacing mặc định; mọi hàm nhận `spacing` theo đúng thứ tự trục của mảng truyền vào.

In [ ]:
def load_mask(case_id):
    img = sitk.ReadImage(str(mask_files[case_id]))
    arr = sitk.GetArrayFromImage(img).astype(np.uint8)          # (z, y, x)
    spacing = tuple(float(s) for s in img.GetSpacing()[::-1])   # dao de khop (z, y, x)
    return arr, spacing

cid0 = cohort["case_id"].iloc[0]
arr, sp = load_mask(cid0)
print(cid0, "| shape", arr.shape, "| spacing", sp, "| labels", np.unique(arr))

## 2) Chạy thử 1 ca: thời gian + QC trực quan (làm trước khi tin số)

Nhìn gì ở overlay: **đỏ** (FCL) phải nằm *bên trong* mảng sụn; **xanh** (footprint) không được bò lên thân xương
hay vào hõm liên lồi cầu. Nếu đỏ chạy dọc *toàn bộ* rìa sụn → nghi khe xương–sụn trong mask > `gap_mm`, kiểm lại mask.
`qc_flipfrac_*` phải ≈ 0 hoặc ≈ 1 (quy ước pháp tuyến nhất quán); ≈ 0.5 là có gì đó sai.

In [ ]:
import matplotlib.pyplot as plt

def qc_overlay(case_id, arr, sp, surf, out_png=None):
    """Lat cat co nhieu FCL nhat: trai = mask; phai = mask + footprint (xanh) + FCL (do)."""
    fcl_map = np.zeros(arr.shape, bool); fp_map = np.zeros(arr.shape, bool)
    for s in surf.values():
        fcl_map |= BM.paint_vertices(s["verts"], s["fcl"], arr.shape, sp)
        fp_map |= BM.paint_vertices(s["verts"], s["footprint"], arr.shape, sp)
    src = fcl_map if fcl_map.any() else fp_map
    per_axis = [src.sum(axis=tuple(a for a in range(3) if a != ax)) for ax in range(3)]
    ax_ = int(np.argmax([c.max() for c in per_axis]))
    z = int(per_axis[ax_].argmax())
    take = lambda v: np.take(v, z, axis=ax_).T
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(take(arr), cmap="nipy_spectral", vmin=0, vmax=8, origin="lower")
    axes[0].set_title(f"{case_id} - mask")
    axes[1].imshow(take(arr), cmap="gray", vmin=0, vmax=8, origin="lower")
    fp2, fcl2 = take(fp_map).astype(float), take(fcl_map).astype(float)
    axes[1].imshow(np.ma.masked_where(fp2 == 0, fp2), cmap="Greens", alpha=0.6, origin="lower", vmin=0, vmax=1)
    axes[1].imshow(np.ma.masked_where(fcl2 == 0, fcl2), cmap="Reds", alpha=0.9, origin="lower", vmin=0, vmax=1)
    axes[1].set_title(f"axis {ax_} slice {z}: footprint (xanh) + FCL (do)")
    for a in axes:
        a.axis("off")
    plt.tight_layout()
    if out_png:
        plt.savefig(out_png, dpi=120)
    plt.show()

t0 = time.time()
res0, surf0 = BM.all_biomarkers(arr, sp, return_surfaces=True, **PARAMS)
dt = time.time() - t0
print(f"{cid0}: {dt:.1f}s/ca -> uoc tinh ca cohort ~{dt * len(cohort) / 3600:.1f} gio CPU (co checkpoint, chay lai tiep duoc)")
s = pd.Series(res0)
display(s[s.index.str.match(r"^(fcl_|thc_|tab_|cab_|qc_)")].round(3).to_frame("value"))
qc_overlay(cid0, arr, sp, surf0, QC_DIR / f"{cid0}_fcl.png")

## 3) Vòng lặp toàn cohort — resumable, checkpoint mỗi `SAVE_EVERY` ca

Đứt Colab thì chạy lại cell này: chỉ làm ca còn thiếu. Ca lỗi được ghi vào `errors.csv`, không dừng vòng lặp.

In [ ]:
if CHECKPOINT_CSV.exists():
    done_df = pd.read_csv(CHECKPOINT_CSV)
    done = set(done_df["case_id"])
else:
    done_df, done = pd.DataFrame(), set()
todo = [c for c in cohort["case_id"] if c not in done]
if MAX_CASES_THIS_RUN:
    todo = todo[:MAX_CASES_THIS_RUN]
print(f"da xong {len(done)} | chay them {len(todo)} / con thieu {len(cohort) - len(done)}")

rows, errors = [], []
pbar = tqdm(todo, unit="ca")
for i, cid in enumerate(pbar):
    try:
        a, s_ = load_mask(cid)
        t0 = time.time()
        r = BM.all_biomarkers(a, s_, **PARAMS)
        r["case_id"] = cid
        r["qc_sec"] = round(time.time() - t0, 1)
        r["qc_spacing"] = str(tuple(round(v, 4) for v in s_))
        rows.append(r)
    except Exception as e:                      # ghi lai, khong dung vong lap
        errors.append((cid, repr(e)))
        pbar.write(f"LOI {cid}: {e!r}")
        continue
    pbar.set_postfix(xong=len(rows), loi=len(errors))
    if (i + 1) % SAVE_EVERY == 0:
        pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(CHECKPOINT_CSV, index=False)

biom = pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).drop_duplicates("case_id", keep="last")
biom.to_csv(CHECKPOINT_CSV, index=False)
print(f"tong {len(biom)}/{len(cohort)} ca | loi lan nay {len(errors)}")
if errors:
    pd.DataFrame(errors, columns=["case_id", "error"]).to_csv(OUT_DIR / "errors.csv", index=False)

## 4) Ghép KL + `source_dataset`, lưu bảng v2, đối chiếu cột cũ với bảng S3

Cột legacy dùng **cùng định nghĩa** với S3 nên phải ra **cùng số** (sai số float). Lệch lớn ⇒ mask trong `MASK_DIR`
đã đổi từ lúc chạy S3, hoặc định nghĩa bị trôi — phải tìm ra trước khi dùng bảng.

In [ ]:
final = cohort.merge(biom, on="case_id", how="inner")
assert not final["case_id"].duplicated().any(), "case_id trung sau merge"
final.to_csv(OUT_CSV, index=False)
print("da luu:", OUT_CSV, "| shape", final.shape)

if S3_CSV.exists():
    s3 = pd.read_csv(S3_CSV).drop_duplicates("case_id")
    legacy_cols = [c for c in s3.columns if c.startswith(BM.LEGACY_PREFIXES) and c in final.columns]
    m = s3[["case_id"] + legacy_cols].merge(final[["case_id"] + legacy_cols], on="case_id", suffixes=("_s3", "_v2"))
    diffs = {c: float(np.nanmax(np.abs(m[f"{c}_s3"] - m[f"{c}_v2"]))) if len(m) else np.nan for c in legacy_cols}
    worst = max(diffs, key=lambda k: (np.nan_to_num(diffs[k], nan=-1)))
    print(f"doi chieu {len(m)} ca chung, {len(legacy_cols)} cot legacy | lech lon nhat: {worst} = {diffs[worst]:.3g}")
    if diffs[worst] > 1e-6:
        print("CANH BAO: cot legacy khac bang S3 -> mask da doi hoac dinh nghia troi. Kiem tra truoc khi dung.")
else:
    print("khong thay biomarker_table.csv cua S3 -> bo qua doi chieu")

## 5) Kiểm tra hợp lý: FCL phải tăng theo KL

Không phải kiểm định chính thức, chỉ là sanity check: dAB% trong y văn tăng mạnh ở KL 3–4.
Nếu `fcl_*_pct` **không** tăng theo KL trong khi `denuded_ratio` cũ có tăng, nghi ngờ footprint (mục 6) hoặc mask.

In [ ]:
from scipy.stats import spearmanr
feat = [c for c in ["fcl_mt_pct", "fcl_lt_pct", "fcl_fem_pct", "thc_tab_mt_mm", "thc_tab_lt_mm", "thc_tab_fem_mm",
                    "thin_le10_mt_pct", "denuded_ratio_tibial", "thickness_med_tib_mm"] if c in final.columns]
print("trung vi theo KL:")
display(final.groupby("KL")[feat].median().round(3))
rho = {c: spearmanr(final["KL"], final[c], nan_policy="omit")[0] for c in feat}
print("Spearman voi KL:")
display(pd.Series(rho).round(3).to_frame("rho"))
has = final[[f"fcl_{c}_pct" for c in ("mt", "lt", "fem")]].gt(0).any(axis=1)
print("ty le ca co FCL > 0 theo KL:")
display(has.groupby(final["KL"]).mean().round(2).to_frame("frac_fcl_gt_0"))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_, c in zip(axes, ["fcl_mt_pct", "fcl_fem_pct", "thc_tab_mt_mm"]):
    final.boxplot(column=c, by="KL", ax=ax_)
    ax_.set_title(c); ax_.set_xlabel("KL")
plt.suptitle(""); plt.tight_layout(); plt.savefig(OUT_DIR / "fcl_vs_kl.png", dpi=130); plt.show()

## 6) (Tùy chọn) Độ nhạy theo bán kính closing

Closing lấp lỗ đường kính < ~2R. R nhỏ bỏ sót ổ lớn; R lớn có thể lấp cả hõm/rìa thật. Đo trên `SENS_N` ca
với R = 5 / 8 / 12 mm rồi ghi giá trị dùng trong báo cáo. Đặt `SENS_N = 0` để bỏ qua.

In [ ]:
SENS_N = 0          # vd 20 (moi ca chay them 2 lan surface_biomarkers)
if SENS_N:
    rows = []
    for cid in tqdm(cohort["case_id"].iloc[:SENS_N].tolist()):
        a, s_ = load_mask(cid)
        for R in (5.0, 8.0, 12.0):
            r = BM.surface_biomarkers(a, s_, **{**PARAMS, "close_mm": R})
            rows.append(dict(case_id=cid, close_mm=R, **{k: r[k] for k in
                        ("fcl_mt_pct", "fcl_lt_pct", "fcl_fem_pct", "tab_mt_mm2", "tab_fem_mm2")}))
    sens = pd.DataFrame(rows)
    display(sens.groupby("close_mm").median().round(2))
    sens.to_csv(OUT_DIR / "closing_sensitivity.csv", index=False)

## 7) Nhiễm gai xương — việc cần làm tiếp (chưa chạy ở đây)

Mask không có nhãn gai xương. Gai xương trưởng thành vào nhãn xương; mũ sụn gai xương dễ vào nhãn sụn ⇒ `tab_*`, `cab_*`
bị kéo ra rìa ở gối OA nặng, ngược chiều mất sụn. Cách kiểm rẻ nhất: file `KXR_SQ_BU00.txt` mà S1 đọc KL cũng có
điểm gai xương X-quang theo khoang (các cột dạng `OSFM/OSFL/OSTM/OSTL`, in `df_kl_raw.columns` để xác nhận tên).
Trong **cùng mức KL**, nếu `tab_mt_mm2` / `cab_fem_mm2` tương quan dương với điểm gai xương ⇒ biomarker đang bị nhiễm,
và cần chuyển footprint sang atlas gối lành (`bsc/atlas.py` đã có khung `ArticularAtlas`) hoặc chỉ đo vùng chịu lực trung tâm.

In [ ]:
KL_FILE = None     # vd "/content/drive/MyDrive/.../KXR_SQ_BU00.txt" - dat de kiem nhiem gai xuong
if KL_FILE and Path(KL_FILE).exists():
    raw = pd.read_csv(KL_FILE, sep=None, engine="python")
    cand = [c for c in raw.columns if "OS" in c.upper() and c.upper()[-2:] in ("FM", "FL", "TM", "TL")]
    print("cot gai xuong ung vien:", cand)
    print("=> join theo subject/side nhu S1 (parse_oai_code), roi trong tung KL: spearman(tab_mt_mm2, OSTM)")
else:
    print("bo qua: dat KL_FILE de kiem nhiem gai xuong")

## Ghi chú
- `biomarker_table_v2.csv` là input của **S7** (`biomarker_s7_ordinal.ipynb`). Bảng S3 cũ giữ nguyên, S4/S5 vẫn chạy như trước.
- Cột `qc_*` (số đỉnh, tỉ lệ lật pháp tuyến, thời gian, spacing) không phải feature; S7 chỉ lấy cột có tiền tố trong
  `BM.FEATURE_PREFIXES`.
- Tham số đo nằm ở `s6_fcl/params.json`. Đổi tham số ⇒ chạy ra **thư mục mới**, không ghi đè.
- FCL ở đây là **cận dưới** (chỉ ổ được bao quanh). Muốn bắt cả khoang trơ trụi cần footprint từ atlas gối lành — việc của giai đoạn sau.